# Archaeology ML Classification (PCA + KNN + Random Forest)

This notebook is a ready-to-upload demo project for an **Algorithm Engineer / ML Engineer** application.

It supports two modes:
1. **Use your own CSV** (recommended): load a dataset with a label column.
2. **Generate a synthetic dataset** (fallback): so the notebook runs out-of-the-box.

You will get:
- Data checks + preprocessing
- PCA visualization
- KNN + Random Forest training
- Metrics (Accuracy / Precision / Recall / F1)
- Confusion matrix
- Feature importance (RF)


In [ ]:
# 1) Install (optional if you already have these)
# If running on Colab, uncomment:
# !pip -q install numpy pandas scikit-learn matplotlib

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

print("Ready ✅")


## 2) Load your dataset (CSV)

**Recommended**: put your CSV in the same folder as this notebook.

### Expected format
- One row = one sample
- Numeric feature columns (e.g., elements, peak intensities, etc.)
- A label column, e.g. `label` (class name) or `y`

If you don't have a CSV at hand, we will generate a synthetic dataset in the next cell.


In [ ]:
# --- Option A: Load your own CSV ---
# Edit these two variables for your dataset:
CSV_PATH = "data.csv"      # your file name
LABEL_COL = "label"        # your label column name

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print("Loaded:", CSV_PATH)
    display(df.head())
else:
    df = None
    print(f"CSV not found at '{CSV_PATH}'. We'll generate a synthetic dataset next.")


## 3) Generate a synthetic dataset (fallback)
If you didn't upload a CSV, this cell creates a dataset with 3 classes and 20 numeric features.
This is just for demo so the whole pipeline runs.


In [ ]:
from sklearn.datasets import make_classification

if df is None:
    X, y = make_classification(
        n_samples=800,
        n_features=20,
        n_informative=10,
        n_redundant=4,
        n_classes=3,
        class_sep=1.4,
        random_state=42
    )
    feature_names = [f"f{i:02d}" for i in range(X.shape[1])]
    df = pd.DataFrame(X, columns=feature_names)
    df["label"] = y
    LABEL_COL = "label"
    print("Synthetic dataset created ✅")
    display(df.head())


## 4) Basic checks + split features/labels
We:
- Drop non-numeric columns except label
- Handle missing values (simple strategy: fill with median)
- Split train/test


In [ ]:
# Make a copy to keep original
data = df.copy()

# Ensure label exists
assert LABEL_COL in data.columns, f"Label column '{LABEL_COL}' not found. Columns: {list(data.columns)}"

# Separate
y = data[LABEL_COL]
X = data.drop(columns=[LABEL_COL])

# Keep only numeric features
numeric_cols = X.select_dtypes(include=[np.number]).columns
X = X[numeric_cols]

# Fill missing values with median (simple & robust)
X = X.fillna(X.median(numeric_only=True))

print("Samples:", X.shape[0])
print("Features:", X.shape[1])
print("Classes:", sorted(pd.Series(y).unique().tolist()))

# Train/test split (stratified is important for classification)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)


## 5) PCA visualization (2D)
PCA helps quickly inspect whether classes have separable structure.


In [ ]:
pca_vis = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2, random_state=42))
])

X_2d = pca_vis.fit_transform(X_train)

plt.figure(figsize=(7, 5))
for cls in sorted(pd.Series(y_train).unique()):
    idx = (pd.Series(y_train).values == cls)
    plt.scatter(X_2d[idx, 0], X_2d[idx, 1], s=18, alpha=0.7, label=str(cls))

plt.title("PCA (2D) on Training Set")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Class", loc="best")
plt.tight_layout()
plt.show()


## 6) Model 1 — KNN (with scaling)
KNN is distance-based, so scaling is required.


In [ ]:
knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=9))
])

knn_model.fit(X_train, y_train)
knn_pred = knn_model.predict(X_test)

print("KNN Results")
print(classification_report(y_test, knn_pred, digits=4))

cm = confusion_matrix(y_test, knn_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(values_format="d")
plt.title("KNN Confusion Matrix")
plt.tight_layout()
plt.show()


## 7) Model 2 — Random Forest
Random Forest is a strong baseline for tabular data and provides feature importance.


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("Random Forest Results")
print(classification_report(y_test, rf_pred, digits=4))

cm = confusion_matrix(y_test, rf_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(values_format="d")
plt.title("Random Forest Confusion Matrix")
plt.tight_layout()
plt.show()


## 8) Feature importance (Random Forest)
This is a quick interpretability view for which variables drive classification.


In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

top_k = 15
plt.figure(figsize=(8, 5))
importances.head(top_k).iloc[::-1].plot(kind="barh")
plt.title(f"Top {top_k} Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

display(importances.head(20).to_frame("importance"))


## 9) Save the best model (optional)
If you want to reuse the trained model later (e.g., for a demo API), you can save it.


In [ ]:
import joblib

OUT_PATH = "best_model.joblib"

# Simple selection rule: pick RF by default (usually stronger on tabular)
best_model = rf_model
joblib.dump(best_model, OUT_PATH)

print("Saved:", OUT_PATH)


## 10) Notes for your GitHub README
Put these bullet points into README.md:
- Built an end-to-end ML pipeline for classification (cleaning, scaling, PCA visualization, model training/evaluation)
- Compared KNN vs Random Forest; reported precision/recall/F1 and confusion matrix
- Provided feature importance for interpretability

If you upload your own dataset later, just replace `data.csv` and set `LABEL_COL`.
